In [1]:
###MODELLING

In [3]:
# import libraries
import pandas as pd
import numpy as np

# Preprocessing
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Train/test split
from sklearn.model_selection import train_test_split

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

# Evaluation
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
)

# Plotting (for ROC curves, feature importance, etc.)
import matplotlib.pyplot as plt

In [ ]:
# read in dfs

model_df_binary = pd.read_parquet("../data/model_df_binary.parquet")
model_df_multiclass = pd.read_parquet("../data/model_df_multiclass.parquet")



# test train split time

In [22]:
# Remove the leftover binary target from the multiclass dataframe to remove risk of leak

model_df_multiclass = model_df_multiclass.drop(columns=['outcome_binary'])

KeyError: "['outcome_binary'] not found in axis"

In [ ]:
print(model_df_binary.columns.tolist())
print(model_df_multiclass.columns.tolist())

['crime_id', 'month', 'reported_by', 'longitude', 'latitude', 'location', 'lsoa_code', 'lsoa_name', 'crime_type', 'last_outcome_category', 'outcome_binary', 'employ_score_rate', 'income_score_rate', 'living_environment_score', 'barriers_score', 'strat_col']
['crime_id', 'month', 'reported_by', 'longitude', 'latitude', 'location', 'lsoa_code', 'lsoa_name', 'crime_type', 'last_outcome_category', 'employ_score_rate', 'income_score_rate', 'living_environment_score', 'barriers_score', 'outcome_multiclass_grouped']


In [ ]:
#Build X/y for the binary dataframe. This will leave X with just crime_type and income_score_rate 

X = model_df_binary.drop(columns=[
    'crime_id',
    'month',
    'reported_by',
    'longitude',
    'latitude',
    'location',
    'lsoa_code',
    'lsoa_name',
    'last_outcome_category',
    'outcome_binary',
    'employ_score_rate',
    'living_environment_score',
    'barriers_score',
    'strat_col'
])
y = model_df_binary['outcome_binary']

In [ ]:
# Build X_multi/y_multi for the multiclass dataframe. This should have X_multi ending up with just crime_type and income_score_rate

X_multi = model_df_multiclass.drop(columns=[
    'crime_id',
    'month',
    'reported_by',
    'longitude',
    'latitude',
    'location',
    'lsoa_code',
    'lsoa_name',
    'last_outcome_category',
    'employ_score_rate',
    'living_environment_score',
    'barriers_score',
    'outcome_multiclass_grouped'
])
y_multi = model_df_multiclass['outcome_multiclass_grouped']




In [ ]:
# Note strat_col stays out of X, but remains in model_df_binary so can reference it for stratification.

In [ ]:
# TEST/TRAIN SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=model_df_binary['strat_col']
)

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

outcome_binary
0    0.923074
1    0.076926
Name: proportion, dtype: float64
outcome_binary
0    0.923097
1    0.076903
Name: proportion, dtype: float64


BINARY SPLIT DONE Both results above come out close to the overall 7.69% resolved / 92.31% unresolved split hopefully

In [ ]:
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi, y_multi,
    test_size=0.2,
    random_state=42,
    stratify=y_multi
)

#check counts. count for each outcome category, smallest first. As long as the smallest category has at least 2 rows, the split will succeed 

y_multi.value_counts().sort_values()

outcome_multiclass_grouped
Other                                                      23
Formal action is not in the public interest               499
Further investigation is not in the public interest       917
Offender given a caution                                 1936
Action to be taken by another organisation               3667
Further action is not in the public interest             3996
Local resolution                                         6465
Investigation complete; no suspect identified          100941
Unable to prosecute suspect                            109113
Name: count, dtype: int64

NOTE: with only 23 rows in Other, an 80/20 split puts abouty 18 in train and 5 in test.  ie.e a small sample for yotheur model to learn that category from, and an even smaller one to evaluate it on — so Other might ends up with unstable or unreliable precision/recall numbers later, due to having so few examples . A limitation to be aware of if Other's metrics look erratic.

In [ ]:
# TEST/TRAIN SPLIT FOR MULTICLASS DF

X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi, y_multi,
    test_size=0.2,
    random_state=42,
    stratify=y_multi
)

In [ ]:
print(y_train_multi.value_counts(normalize=True))
print(y_test_multi.value_counts(normalize=True))

outcome_multiclass_grouped
Unable to prosecute suspect                            0.479497
Investigation complete; no suspect identified          0.443583
Local resolution                                       0.028411
Further action is not in the public interest           0.017562
Action to be taken by another organisation             0.016117
Offender given a caution                               0.008509
Further investigation is not in the public interest    0.004032
Formal action is not in the public interest            0.002192
Other                                                  0.000099
Name: proportion, dtype: float64
outcome_multiclass_grouped
Unable to prosecute suspect                            0.479500
Investigation complete; no suspect identified          0.443597
Local resolution                                       0.028410
Further action is not in the public interest           0.017556
Action to be taken by another organisation             0.016106
Offender given a 

So at this point you have, fully built and checked:
BinaryMulticlassFeaturesX (crime_type, income_score_rate)X_multi (same two columns)Targety (outcome_binary)y_multi (outcome_multiclass_grouped)SplitX_train, X_test, y_train, y_testX_train_multi, X_test_multi, y_train_multi, y_test_multiStratified oncrime_type + outcome_binary combinedoutcome_multiclass_grouped aloneVerifiedproportions match to ~0.01%proportions match to ~0.01%

Multiclass split done - The above shows a clean match every category's proportion in train and test lines up almost exactly (down to the 4th decimal place), including the tiny Other category at ~0.01%. The stratification worked correctly across all nine outcome categories


THE LIST OF NEW VARIABLES - because I always forget!

**Binary dataframe (`model_df_binary`)**

| Variable | Contents |
|---|---|
| `X` | Features: `crime_type`, `income_score_rate` |
| `y` | Target: `outcome_binary` |
| `X_train` | Training features (80%) |
| `X_test` | Test features (20%) |
| `y_train` | Training target (80%) |
| `y_test` | Test target (20%) |

Stratified on: `model_df_binary['strat_col']` (`crime_type` + `outcome_binary` combined)

**Multiclass dataframe (`model_df_multiclass`)**

| Variable | Contents |
|---|---|
| `X_multi` | Features: `crime_type`, `income_score_rate` |
| `y_multi` | Target: `outcome_multiclass_grouped` |
| `X_train_multi` | Training features (80%) |
| `X_test_multi` | Test features (20%) |
| `y_train_multi` | Training target (80%) |
| `y_test_multi` | Test target (20%) |

Stratified on: `y_multi` alone (`outcome_multiclass_grouped`)

# One hot encoding - crime_type

In [23]:
# need to encode it on X_train first, then apply the same encoding to X_test.

# one-hot encode crime_type on training data
X_train_encoded = pd.get_dummies(X_train, columns=['crime_type'], drop_first=True)

# apply the same encoding to test data, aligning columns to match X_train_encoded
X_test_encoded = pd.get_dummies(X_test, columns=['crime_type'], drop_first=True)
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

# NOTE: The reindex(columns=X_train_encoded.columns, fill_value=0) line is the important safety step — it forces X_test_encoded to have exactly the same dummy columns as X_train_encoded, in the same order, filling in 0 for any category that didn't happen to appear in the test set. Without this, if test has a crime type train doesn't (or vice versa), your two encoded dataframes could end up with different numbers of columns, which would break the model when you try to predict.

In [ ]:
# check the shapes match up
X_train_encoded.shape[1] == X_test_encoded.shape[1]

True

# scale income_score_rate for use in logistic regression 

In [25]:
from sklearn.preprocessing import StandardScaler

# instantiate the scaler
scaler = StandardScaler()

# fit on training data only, then transform it
X_train_encoded['income_score_rate'] = scaler.fit_transform(X_train_encoded[['income_score_rate']])

# use the already-fitted scaler to transform test data (no fitting here)
X_test_encoded['income_score_rate'] = scaler.transform(X_test_encoded[['income_score_rate']])

In [26]:
print(X_train_encoded['income_score_rate'].mean(), X_train_encoded['income_score_rate'].std())
print(X_test_encoded['income_score_rate'].mean(), X_test_encoded['income_score_rate'].std())

-8.368234542201722e-17 1.0000027465698773
0.0032043355275436607 0.9988118316778521
